# SeraTune TRIBE v2 Worker

**4 cells — run top to bottom.**

| Cell | What it does | Re-run? |
|------|-------------|---------|
| 1 | Install packages | Once per session |
| 2 | **Restart runtime** | Once (after Cell 1) |
| 3 | Load TRIBE model into GPU memory | Once |
| 4 | Start server + tunnel | Re-run anytime (model stays loaded) |

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** (or A100 with Colab Pro)
- Have your [HuggingFace token](https://huggingface.co/settings/tokens) ready (accept [LLaMA 3.2-3B license](https://huggingface.co/meta-llama/Llama-3.2-3B) first)
- No tunnel token needed — we use [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) (free, no sign-up)

In [ ]:
# ============================================================
# CELL 1: Install everything (run once per session)
# ============================================================
!pip uninstall -y torch torchaudio torchvision numpy neuralset neuraltrain tribev2 click gtts 2>/dev/null

!pip install -q \
  "numpy==2.2.6" \
  "torch==2.6.0" \
  "torchaudio==2.6.0" \
  "torchvision==0.21.0" \
  --index-url https://download.pytorch.org/whl/cu124

!pip install -q \
  "neuralset==0.0.2" \
  "neuraltrain==0.0.2" \
  "x_transformers==1.27.20" \
  "moviepy>=2.2.1" \
  gtts julius Levenshtein transformers huggingface_hub \
  soundfile librosa langdetect spacy

!pip install -q git+https://github.com/facebookresearch/tribev2.git
!pip install -q fastapi uvicorn python-multipart httpx nest_asyncio

# Install cloudflared (Cloudflare Tunnel) — no account needed
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print('\n✅ Packages installed. Now run Cell 2 to restart the runtime.')

In [ ]:
# ============================================================
# CELL 2: Restart runtime so fresh numpy/torch are picked up
# ============================================================
# After this, skip straight to Cell 3 (do NOT re-run Cell 1).
import os
os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# CELL 3: Load TRIBE model (run once — stays in memory)
# ============================================================
# After the runtime restart, this cell loads the model into GPU
# memory. It stays loaded even if you re-run Cell 4.

HF_TOKEN = ""       # paste your HuggingFace token here

if not HF_TOKEN:
    HF_TOKEN = input('Enter your HuggingFace token: ')

import os
os.environ["HF_TOKEN"] = HF_TOKEN

import torch
from huggingface_hub import login
from tribev2 import TribeModel

login(token=HF_TOKEN)

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A"
print(f"GPU: {gpu} ({vram})")

print("Loading TRIBE v2 model...")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print(f"✅ Model loaded! Now run Cell 4 to start the server.")

In [ ]:
# ============================================================
# CELL 4: Start server + tunnel (re-run anytime without
#          reloading the model)
# ============================================================
# Interrupt this cell (stop button) and re-run it to restart
# the server. The model stays in GPU memory from Cell 3.

import nest_asyncio
nest_asyncio.apply()

import sys, os, logging, tempfile, time, io, base64, gzip, uuid, threading
import subprocess, re
import numpy as np
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse
import uvicorn

logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger(__name__)

# ── Job queue ───────────────────────────────────────────────────────────────
jobs: dict[str, dict] = {}
job_queue: list[str] = []
queue_lock = threading.Lock()

def _process_queue():
    """Background thread: process jobs one at a time."""
    while True:
        job_id = None
        with queue_lock:
            if job_queue:
                job_id = job_queue.pop(0)
        if job_id is None:
            time.sleep(0.5)
            continue

        job = jobs[job_id]
        job["status"] = "processing"
        tmp_path = job.get("tmp_path")
        try:
            logger.info("[%s] Starting inference on %s...", job_id, job.get("filename"))
            start = time.time()
            df = model.get_events_dataframe(audio_path=tmp_path)
            preds, segments = model.predict(events=df)
            elapsed = time.time() - start
            compressed = _compress_preds(preds)
            raw_mb = preds.nbytes / 1e6
            comp_mb = len(compressed) / 1e6
            logger.info(
                "[%s] Inference complete: %s -> %s in %.1fs (%.1fMB -> %.1fMB)",
                job_id, job.get("filename"), preds.shape, elapsed, raw_mb, comp_mb,
            )
            job["result"] = {
                "preds_b64gz": compressed,
                "shape": list(preds.shape),
                "n_segments": preds.shape[0],
                "n_vertices": preds.shape[1],
                "inference_time_s": round(elapsed, 2),
            }
            job["status"] = "completed"
        except Exception as e:
            logger.exception("[%s] Inference failed", job_id)
            job["status"] = "failed"
            job["error"] = str(e)
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.unlink(tmp_path)

worker_thread = threading.Thread(target=_process_queue, daemon=True)
worker_thread.start()

# FastAPI app
app = FastAPI(title="TRIBE v2 Worker")

@app.get("/health")
async def health():
    with queue_lock:
        pending = len(job_queue)
    processing = sum(1 for j in jobs.values() if j["status"] == "processing")
    return {
        "status": "ok",
        "model_loaded": True,
        "jobs_pending": pending,
        "jobs_processing": processing,
    }

def _compress_preds(preds: np.ndarray) -> str:
    """Gzip + base64 encode a numpy array for efficient transfer."""
    buf = io.BytesIO()
    np.save(buf, preds.astype(np.float16))
    compressed = gzip.compress(buf.getvalue(), compresslevel=1)
    return base64.b64encode(compressed).decode("ascii")

@app.post("/submit")
async def submit_job(audio: UploadFile = File(...)):
    """Accept an audio file and queue it for inference."""
    job_id = uuid.uuid4().hex[:12]
    suffix = os.path.splitext(audio.filename or "audio.wav")[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        content = await audio.read()
        tmp.write(content)
        tmp_path = tmp.name

    job = {
        "status": "queued",
        "filename": audio.filename,
        "tmp_path": tmp_path,
        "size_bytes": len(content),
        "submitted_at": time.time(),
        "result": None,
        "error": None,
    }
    jobs[job_id] = job
    with queue_lock:
        job_queue.append(job_id)

    logger.info("[%s] Job queued: %s (%d bytes)", job_id, audio.filename, len(content))
    return {"job_id": job_id, "status": "queued"}


@app.get("/status/{job_id}")
async def job_status(job_id: str):
    """Poll for job result."""
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(404, f"Job {job_id} not found")

    resp = {"job_id": job_id, "status": job["status"]}

    if job["status"] == "completed":
        resp["result"] = job["result"]
        if not job.get("_cleanup_scheduled"):
            job["_cleanup_scheduled"] = True
            import asyncio
            asyncio.get_event_loop().call_later(60, lambda jid=job_id: jobs.pop(jid, None))
    elif job["status"] == "failed":
        resp["error"] = job["error"]
        if not job.get("_cleanup_scheduled"):
            job["_cleanup_scheduled"] = True
            import asyncio
            asyncio.get_event_loop().call_later(60, lambda jid=job_id: jobs.pop(jid, None))
    elif job["status"] == "queued":
        with queue_lock:
            try:
                resp["queue_position"] = job_queue.index(job_id) + 1
            except ValueError:
                resp["queue_position"] = 0

    return resp


# ── Start Cloudflare tunnel ─────────────────────────────────────────────────
print('Starting Cloudflare tunnel...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8001'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
url_event = threading.Event()

def _read_tunnel_output():
    global public_url
    for line in tunnel_proc.stdout:
        m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if m and not public_url:
            public_url = m.group(1)
            url_event.set()

tunnel_thread = threading.Thread(target=_read_tunnel_output, daemon=True)
tunnel_thread.start()

url_event.wait(timeout=30)
if not public_url:
    print('ERROR: Could not get tunnel URL from cloudflared.')
    tunnel_proc.terminate()
    raise RuntimeError('Cloudflare tunnel failed to start')

print()
print('=' * 60)
print(f'Cloudflare tunnel ready!')
print(f'')
print(f'  Public URL: {public_url}')
print(f'')
print(f'  Add to your backend .env:')
print(f'    TRIBE_WORKER_URL={public_url}')
print(f'    USE_MOCK_TRIBE=false')
print('=' * 60)
print()
print('Server starting... (interrupt and re-run this cell to restart)')
print('Model stays loaded in memory from Cell 3.\n')

# Run server (blocks — interrupt to stop, then re-run this cell)
try:
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")
except KeyboardInterrupt:
    tunnel_proc.terminate()
    print('\nServer stopped. Re-run this cell to restart (model is still loaded).')